In [ ]:
import pandas as pd

pesquisador = 'titulados'

ano = '2023'

df_filtrado = pd.read_csv(f"df_mapeamento_{pesquisador}_{ano}_filtrado.csv")

df_titulados = pd.read_parquet(f"df_{pesquisador}_{ano}.parquet")

df_programas = pd.read_parquet(f"df_programas_{ano}.parquet")

df_filtrado.head()

In [72]:
mapa_regiao = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}


In [73]:
def gerar_df_formandos_por_uf(df_data, df_titulados=None):
    """Cria DataFrame com quantidade de formandos únicos por UF_PESQUISADOR com afinidade ALTA ou MEDIA.

    Quando `df_titulados` é fornecido, diferencia MESTRADO e DOUTORADO usando NM_GRAU_ACADEMICO.
    """
    df_copy = df_data.copy()

    # Filtrar apenas afinidade ALTA ou MEDIA
    df_filtro = df_copy[df_copy['nivel'].isin(['ALTA', 'MEDIA'])]

    # Contar formandos únicos por HASH_ID e UF
    df_base = df_filtro[['HASH_ID', 'UF_PESQUISADOR']].drop_duplicates()

    df_grau = df_titulados[['HASH_ID', 'NM_GRAU_ACADEMICO']].copy()
    grau = df_grau['NM_GRAU_ACADEMICO']
    df_grau.loc[grau.str.contains('MESTRADO', na=False), 'NM_GRAU_ACADEMICO'] = 'MESTRADO'
    df_grau.loc[grau.str.contains('DOUTORADO', na=False), 'NM_GRAU_ACADEMICO'] = 'DOUTORADO'
    
    df_merge = df_base.merge(df_grau, on='HASH_ID', how='left')

    df_result = (
        df_merge.groupby(['UF_PESQUISADOR', 'NM_GRAU_ACADEMICO'])['HASH_ID']
        .nunique()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ['MESTRADO', 'DOUTORADO']:
        if col not in df_result.columns:
            df_result[col] = 0

    df_result = df_result.rename(columns={
        'MESTRADO': 'FORMANDOS_MESTRADO',
        'DOUTORADO': 'FORMANDOS_DOUTORADO'
    })
    df_result['FORMANDOS_COM_AFINIDADE'] = (df_result[['FORMANDOS_MESTRADO', 'FORMANDOS_DOUTORADO']].sum(axis=1))
    
    coluna_ordem = ['REGIAO', 'UF_PESQUISADOR', 'FORMANDOS_MESTRADO', 'FORMANDOS_DOUTORADO', 'FORMANDOS_COM_AFINIDADE']

    # Adicionar região
    df_result['REGIAO'] = df_result['UF_PESQUISADOR'].map(mapa_regiao)

    # Reordenar colunas
    df_result = df_result[[c for c in coluna_ordem if c in df_result.columns]]

    # Adicionar total por região
    region_totals = (
        df_result
        .groupby('REGIAO')[[c for c in coluna_ordem if c not in ['REGIAO', 'UF_PESQUISADOR'] and c in df_result.columns]]
        .sum()
        .reset_index()
    )
    region_totals['UF_PESQUISADOR'] = 'TOTAL'
    region_totals = region_totals[[c for c in coluna_ordem if c in region_totals.columns]]

    df_result = pd.concat([df_result, region_totals], ignore_index=True)

    # Ordenar por região e UF
    df_result['UF_ORD'] = df_result['UF_PESQUISADOR'].where(df_result['UF_PESQUISADOR'] != 'TOTAL', 'ZZZZ')
    df_result = df_result.sort_values(['REGIAO', 'UF_ORD'])

    return df_result[[c for c in coluna_ordem if c in df_result.columns]]

In [74]:
df_formandos_por_uf = gerar_df_formandos_por_uf(df_filtrado, df_titulados)
df_formandos_por_uf.to_csv('formandos_por_uf_' + ano + '.csv', index=False)
df_formandos_por_uf

NM_GRAU_ACADEMICO,REGIAO,UF_PESQUISADOR,FORMANDOS_MESTRADO,FORMANDOS_DOUTORADO,FORMANDOS_COM_AFINIDADE
6,Centro-Oeste,DF,283,91,374
8,Centro-Oeste,GO,186,66,252
11,Centro-Oeste,MS,182,61,243
12,Centro-Oeste,MT,62,8,70
27,Centro-Oeste,TOTAL,713,226,939
1,Nordeste,AL,127,16,143
4,Nordeste,BA,262,106,368
5,Nordeste,CE,414,93,507
9,Nordeste,MA,93,15,108
14,Nordeste,PB,202,66,268


In [75]:
df_programas_CC = df_programas[df_programas.NM_AREA_CONHECIMENTO == 'CIÊNCIA DA COMPUTAÇÃO']
df_programas_CC.to_csv('programas_Computacao_' + ano + '.csv', index=False)
df_programas_CC[['NM_PROGRAMA_IES','NM_GRANDE_AREA_CONHECIMENTO', 'NM_AREA_CONHECIMENTO']]

,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_AREA_CONHECIMENTO
3,TECNOLOGIA DA INFORMAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
71,CIÊNCIA DA COMPUTAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
201,COMPUTAÇÃO APLICADA,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
246,CIÊNCIA DA COMPUTAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
278,CIÊNCIA DA COMPUTAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
...,...,...,...
4472,CIÊNCIA DA COMPUTAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
4518,CIÊNCIA DA COMPUTAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
4580,CIÊNCIA DA COMPUTAÇÃO,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO
4601,ENGENHARIA DE SOFTWARE,CIÊNCIAS EXATAS E DA TERRA,CIÊNCIA DA COMPUTAÇÃO


In [76]:
def gerar_df_nivel_por_programa_unico(df_filtrado, df_titulados, df_programas):
    """
    Gera a quantidade de pesquisadores/titulados únicos por programa
    que tiveram afinidade MEDIA ou ALTA.
    
    Cada HASH_ID é contado apenas uma vez por programa.
    """

    df_copy = df_filtrado.copy()

    # Filtrar apenas afinidade ALTA ou MEDIA
    df_filtro = df_copy[df_copy['nivel'].isin(['ALTA', 'MEDIA'])].copy()

    # Manter apenas pesquisadores únicos que tiveram MEDIA ou ALTA
    df_base = df_filtro[['HASH_ID']].drop_duplicates()

    # Pegar programa de cada titulado
    df_prog = df_titulados[['HASH_ID', 'CD_PROGRAMA']].drop_duplicates()

    # Associar pesquisador ao programa
    df_merge = df_base.merge(df_prog, on='HASH_ID', how='left')

    # Evitar contar o mesmo pesquisador mais de uma vez no mesmo programa
    df_merge = df_merge[['HASH_ID', 'CD_PROGRAMA']].drop_duplicates()

    # Dados dos programas
    df_programas_copy = df_programas[
        [
            'CD_PROGRAMA_IES',
            'NM_PROGRAMA_IES',
        ]
    ].copy()

    df_programas_copy = df_programas_copy.rename(columns={
        'NM_PROGRAMA_IES': 'NM_PROGRAMA'
    })

    # Juntar metadados do programa
    df_merge = df_merge.merge(
        df_programas_copy,
        left_on='CD_PROGRAMA',
        right_on='CD_PROGRAMA_IES',
        how='left'
    )

    # Contar pesquisadores únicos por programa
    df_result = (
        df_merge
        .groupby(
            [
                'CD_PROGRAMA',
                'NM_PROGRAMA',
            ],
            dropna=False
        )['HASH_ID']
        .nunique()
        .reset_index(name='QTD_PESQUISADORES_MEDIA_ALTA')
    )

    return df_result

df_nivel_por_programa_unico = gerar_df_nivel_por_programa_unico(df_filtrado, df_titulados, df_programas)
df_nivel_por_programa_unico.sort_values('QTD_PESQUISADORES_MEDIA_ALTA', ascending=False)

,CD_PROGRAMA,NM_PROGRAMA,QTD_PESQUISADORES_MEDIA_ALTA
751,30004012074P8,EDUCAÇÃO PROFISSIONAL E TECNOLÓGICA,123
1098,31102000001P6,PROFNIT - PROPRIEDADE INTELECTUAL E TRANSFERÊN...,89
449,25001019004P6,CIÊNCIAS DA COMPUTAÇÃO,81
2907,53045009001P3,ADMINISTRAÇÃO PÚBLICA EM REDE NACIONAL,74
1487,33002010097P3,ARQUITETURA E URBANISMO,72
...,...,...,...
1984,33142017003P3,GOVERNANÇA CORPORATIVA,1
1989,33144010006P9,NANOCIÊNCIAS E MATERIAIS AVANÇADOS,1
1995,33144010014P1,ENGENHARIA ELÉTRICA,1
1998,33144010018P7,CIÊNCIA E TECNOLOGIA AMBIENTAL,1


In [77]:
def calcular_participacao_alta_media(df_nivel_por_programa_unico, df_programas_CC):
    """Calcula a participação total de ALTA+MEDIA dos programas CC versus demais."""
    programas_cc = set(df_programas_CC['CD_PROGRAMA_IES'].astype(str))
    df_result = df_nivel_por_programa_unico.copy()
    df_result['GRUPO'] = df_result['CD_PROGRAMA'].apply(
        lambda x: 'CC' if x in programas_cc else 'OUTROS'
    )
    resumo = (
        df_result.groupby('GRUPO')['QTD_PESQUISADORES_MEDIA_ALTA']
        .sum()
        .reset_index()
    )
    total = resumo['QTD_PESQUISADORES_MEDIA_ALTA'].sum()
    resumo['PERCENTUAL'] = (resumo['QTD_PESQUISADORES_MEDIA_ALTA'] / total * 100).round(2)
    return resumo

resumo_cc_vs_outros = calcular_participacao_alta_media(df_nivel_por_programa_unico, df_programas_CC)
print(resumo_cc_vs_outros)


    GRUPO  QTD_PESQUISADORES_MEDIA_ALTA  PERCENTUAL
0      CC                          1296        10.4
1  OUTROS                         11163        89.6


In [78]:
def gerar_tabela_grande_area_ia(df_filtrado_ano, df_titulados_ano):
    """Gera resumo por grande area para titulados mapeados em IA (ALTA ou MEDIA) usando merge por HASH_ID."""
    df_map = (
        df_filtrado_ano.loc[df_filtrado_ano['nivel'].isin(['ALTA', 'MEDIA']), ['HASH_ID']]
        .drop_duplicates()
    )

    df_tit = (
        df_titulados_ano[['HASH_ID', 'CD_PROGRAMA', 'NM_GRANDE_AREA_CONHECIMENTO', 'NM_GRAU_ACADEMICO']]
        .dropna(subset=['HASH_ID'])
        .copy()
    )

    grau = df_tit['NM_GRAU_ACADEMICO']
    df_tit.loc[grau.str.contains('MESTRADO', na=False), 'NM_GRAU_ACADEMICO'] = 'MESTRADO'
    df_tit.loc[grau.str.contains('DOUTORADO', na=False), 'NM_GRAU_ACADEMICO'] = 'DOUTORADO'

    df_merge = df_map.merge(df_tit, on='HASH_ID', how='inner')
    df_merge = df_merge.dropna(subset=['NM_GRAU_ACADEMICO'])

    contagem_titulados = (
        df_merge.groupby(['NM_GRANDE_AREA_CONHECIMENTO', 'NM_GRAU_ACADEMICO'])['HASH_ID']
        .nunique()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ['MESTRADO', 'DOUTORADO']:
        if col not in contagem_titulados.columns:
            contagem_titulados[col] = 0

    contagem_ppgs = (
        df_merge.groupby('NM_GRANDE_AREA_CONHECIMENTO')['CD_PROGRAMA']
        .nunique()
        .reset_index(name='PPGS')
    )

    resultado = contagem_titulados.merge(contagem_ppgs, on='NM_GRANDE_AREA_CONHECIMENTO', how='left')
    resultado['PPGS'] = resultado['PPGS'].fillna(0).astype(int)

    return resultado[['NM_GRANDE_AREA_CONHECIMENTO', 'MESTRADO', 'DOUTORADO', 'PPGS']]


In [85]:
df_areas =  gerar_tabela_grande_area_ia(df_filtrado, df_titulados)
df_areas

,NM_GRANDE_AREA_CONHECIMENTO,MESTRADO,DOUTORADO,PPGS
0,CIÊNCIAS AGRÁRIAS,314,249,220
1,CIÊNCIAS BIOLÓGICAS,353,281,160
2,CIÊNCIAS DA SAÚDE,692,407,424
3,CIÊNCIAS EXATAS E DA TERRA,1099,481,216
4,CIÊNCIAS HUMANAS,839,290,387
5,CIÊNCIAS SOCIAIS APLICADAS,2582,785,544
6,ENGENHARIAS,1039,409,303
7,"LINGÜÍSTICA, LETRAS E ARTES",95,29,64
8,MULTIDISCIPLINAR,1484,402,550


In [86]:
df_areas.to_csv(f'grandes_areas_grau_{ano}.csv', index=False)

In [87]:
df_titulados[df_titulados.NM_PRODUCAO.str.contains('HETEROGENEOUS GRAPHS')].NM_PRODUCAO.values[0]

'HETEROGENEOUS GRAPHS FOR TEXT REPRESENTATION: AN INTEGRATED APPROACH WITH LANGUAGE MODELS'

In [88]:
hash_id = df_titulados[df_titulados.NM_PRODUCAO.str.contains('HETEROGENEOUS GRAPHS')].HASH_ID.values[0]

In [90]:
df_hash = df_filtrado[df_filtrado.HASH_ID == hash_id]

In [99]:
from pprint import pprint
pprint(df_hash.iloc[2].Qwen3)

('{"nivel": "ALTA", "explicacao": "O(a) pesquisador(a) atua na área de '
 'Ciências da Computação com foco em Grafos Heterogêneos, Modelos de '
 'Linguagem, Aprendizado de Representação e Mineração de Opiniões — todos '
 'temas diretamente relacionados à Inteligência Artificial e à Ciência de '
 "Dados. O título da tese, 'Heterogeneous Graphs for Text Representation: An "
 "Integrated Approach with Language Models', e o resumo destacam a integração "
 'de grafos heterogêneos com modelos de linguagem para melhorar a '
 'representação de dados complexos, especialmente em textos, o que é uma '
 'aplicação avançada de técnicas de IA e ciência de dados. As palavras-chave '
 "do pesquisador, como 'LANGUAGE MODELS', 'REPRESENTATION LEARNING', 'UNIFIED "
 "EMBEDDING SPACE' e 'OPINION MINING', são descritores específicos e altamente "
 "relevantes para o tema estratégico, especialmente em relação a 'inteligência "
 "artificial', 'big data', 'transformação digital', 'dados abertos, acessíveis "


In [103]:
from pprint import pprint
pprint(df_hash.iloc[0].Qwen3)

('{"nivel": "MEDIA", "explicacao": "O(a) pesquisador(a) atua na área de '
 'Ciência da Computação com foco em grafos heterogêneos, modelos de linguagem, '
 'aprendizado de representação e mineração de opiniões, temas que estão '
 'diretamente relacionados à inteligência artificial e à ciência de dados, que '
 "são descritores específicos do tema estratégico 'Tecnologias da Informação e "
 "Comunicação'. O resumo da tese destaca a integração de grafos heterogêneos "
 'com modelos de linguagem para melhorar a representação de dados complexos, o '
 'que se alinha com a aplicação de técnicas de inteligência artificial e '
 'ciência de dados em sistemas de informação. Embora não haja menção direta a '
 "'alfabetização digital', 'mulheres nas TICs' ou 'saúde digital', os "
 "descritores 'inteligência artificial' e 'ciência de dados' estão claramente "
 'presentes nos dados do pesquisador, especialmente no contexto de aprendizado '
 'de máquina e representação de dados. A proximidade semântic

In [ ]:
import pandas as pd

pesquisador = 'titulados'

df_filtrado23 = pd.read_csv(f"df_mapeamento_{pesquisador}_2023_filtrado.csv")

df_filtrado24 = pd.read_csv(f"df_mapeamento_{pesquisador}_2024_filtrado.csv")

df_filtrado = pd.concat([df_filtrado23, df_filtrado24], ignore_index=True)
df_filtrado.head()

In [33]:
df_filtrado = df_filtrado[df_filtrado['nivel'].isin(['ALTA', 'MEDIA'])]

In [ ]:
import pandas as pd

tabela = pd.crosstab(df_filtrado['TEMA_ID'], df_filtrado["UF_PESQUISADOR"],
    values=df_filtrado["HASH_ID"],
    aggfunc='nunique',
    dropna=False
)
tabela.to_csv('tabela_crosstab_uf_tema.csv')